In [10]:
%%capture
%pip install -e ..

In [ ]:
import os
import yaml
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path
from sycophancy.prompt_builder import PromptBuilder
from sycophancy.verdict_analyser import ResultsAnalyser
from sycophancy.clients.gemini_client import GeminiClient
from sycophancy.clients.ollama_cloud_client import OllamaCloudClient
from sycophancy.clients.ollama_local_client import OllamaLocalClient

In [12]:
DATA_DIR = Path().cwd().parent / "data"

In [13]:
with open(DATA_DIR / "questions.yaml", "r") as file:
    questions = yaml.safe_load(file)

In [14]:
# Uncomment the type of client you want to use
load_dotenv()
# client = GeminiClient(api_key = os.getenv("GEMINI_API_KEY"))
# client = OllamaCloudClient(api_key = os.getenv("OLLAMA_API_KEY"))
client = OllamaLocalClient(model="qwen3:4b")

In [ ]:
builder = PromptBuilder()

outputs = {}
for question_number in tqdm(list(range(5))):
    output = {}

    question = questions[f"question_{question_number}"]["question"]
    output["question"] = question

    prompt1 = builder.build_prompt_with_memory(question, builder.neutral_memory)
    output["answer_neutral"] = client.get_response(prompt1)
    print(output["answer_neutral"])

    prompt2 = builder.build_prompt_with_memory(question, builder.positive_memory)
    output["answer_positive"] = client.get_response(prompt2)
    print(output["answer_positive"])

    prompt3 = builder.build_prompt_with_memory(question, builder.negative_memory)
    output["answer_negative"] = client.get_response(prompt3)
    print(output["answer_negative"])

    outputs[f"question_{question_number}"] = output

    with open(DATA_DIR / "answers" / f"answers_{client.model}_run1_14.yaml", "w") as file:
        yaml.safe_dump(outputs, file)